In [8]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [8]:
print(os.listdir("../data"))

['.gitkeep', '10Y.csv', '3Y.csv', 'benchmark_map.csv', 'bond_return_proxies.csv', 'Brent_OIL.csv', 'CPI.csv', 'CPI_Sentiment.csv', 'credit_spreads.csv', 'DWA_Momentum_ETF.csv', 'DXY_data.csv', 'etf_prices.csv', 'F-F_Momentum_Factor_daily.csv', 'index_prices.csv', 'ISM_Manufacturing.csv', 'ISM_Services.csv', 'M1.csv', 'M2.csv', 'NBER_US.csv', 'processed', 'SPX.csv', 'treasury_yields.csv', 'VIX.csv']


<hr>
Equity Market Dynamics Features
<hr>

In [49]:
# ------------------------------------------------
# 1. Load index price data
# ------------------------------------------------
idx = pd.read_csv("../data/index_prices.csv", header=1)
idx.columns = ["Date"] + list(idx.columns[1:])

print("Columns in index_prices.csv:")
print(idx.columns.tolist())

spx = idx[["Date", "SPX Index"]].copy()
spx.dropna(how='any',inplace=True)

# Convert date and sort
spx["Date"] = pd.to_datetime(spx["Date"])
spx = spx.sort_values("Date").reset_index(drop=True)
spx["SPX Index"] = pd.to_numeric(spx["SPX Index"], errors="coerce")

# ------------------------------------------------
# 3. Compute features
# ------------------------------------------------

# Daily log return
spx["SPX_return"] = np.log(spx["SPX Index"] / spx["SPX Index"].shift(1))
#spx = spx

# 20-day rolling volatility of returns
spx["SPX_volatility_20d"] = spx["SPX_return"].dropna().rolling(20).std()

# 252-day rolling drawdown
rolling_max_252 = spx["SPX Index"].rolling(252, min_periods=1).max()
spx["SPX_drawdown"] = spx["SPX Index"] / rolling_max_252 - 1

# ------------------------------------------------
# Moving Average Features
# ------------------------------------------------

spx["SPX_MA50"] = spx["SPX Index"].rolling(50).mean()
spx["SPX_MA200"] = spx["SPX Index"].rolling(200).mean()

# continuous signal
spx["SPX_MA_signal"] = spx["SPX_MA50"] - spx["SPX_MA200"]


# ------------------------------------------------
# 4. Load Fama-French daily momentum factor
# ------------------------------------------------
mom = pd.read_csv(
    "../data/F-F_Momentum_Factor_daily.csv",
    header = 1,
    skiprows=13,
    skipfooter=1,
    names=["Date","Mom"],
    engine="python"
)


mom["Date"] = pd.to_datetime(mom["Date"], format="%Y%m%d")
mom["Mom"] = mom["Mom"] / 100


mom =  mom[mom.Date> pd.to_datetime('1999-12-31')]

spx = spx.merge(mom, on="Date", how="left")

spx =  spx[spx.Date< pd.to_datetime('2026-01-01')]


# ------------------------------------------------
# Select final Equity Market Dynamics features
# ------------------------------------------------

equity_market_dynamics_features = spx[[
    "Date",
    "SPX_return",
    "SPX_volatility_20d",
    "SPX_drawdown",
    "SPX_MA_signal",
    "Mom"
]].copy()

os.makedirs("../data/processed", exist_ok=True)

equity_market_dynamics_features.to_csv(
    "../data/processed/equity_market_dynamics_features.csv",
    index=False
)

print(equity_market_dynamics_features.head())
print(equity_market_dynamics_features.shape)

Columns in index_prices.csv:
['Date', 'SPX Index', 'RLV Index', 'RU20INTR Index', 'RTY Index', 'LUACTRUU Index', 'LF98TRUU Index', 'GOLDLNPM Index']
        Date  SPX_return  SPX_volatility_20d  SPX_drawdown  SPX_MA_signal  \
0 1999-12-31         NaN                 NaN      0.000000            NaN   
1 2000-01-03   -0.009595                 NaN     -0.009549            NaN   
2 2000-01-04   -0.039099                 NaN     -0.047528            NaN   
3 2000-01-05    0.001920                 NaN     -0.045697            NaN   
4 2000-01-06    0.000955                 NaN     -0.044785            NaN   

      Mom  
0     NaN  
1 -0.0006  
2 -0.0191  
3 -0.0049  
4 -0.0149  
(6540, 6)


In [22]:
#store SPX dates

spx_dates = pd.to_datetime(spx["Date"])


<hr>
Volatility / Risk Sentiment 
<hr>

In [24]:



vix = pd.read_csv("../data/VIX.csv",header=0,names=["Date","VIX"])
vix["Date"] = pd.to_datetime(vix["Date"])
vix = vix.set_index("Date").reindex(spx_dates).reset_index().rename(columns={"index": "Date"})
vix.ffill(inplace=True)
vix['VIXChange'] =  np.log(vix["VIX"] / vix["VIX"].shift(1))
vix['VIX3m-VIX'] = vix['VIX'] * (3**.5) - vix['VIX'] 

vix =  vix[vix.Date< pd.to_datetime('2026-01-01')]

Volatility_Risk_Sentiment_features = vix.copy()


Volatility_Risk_Sentiment_features.to_csv(
    "../data/processed/Volatility_Risk_Sentiment_features.csv",
    index=False
)

print(Volatility_Risk_Sentiment_features.shape)

(6540, 4)


<hr>
Interest Rate Environment 
<hr>

In [27]:
import pandas as pd
yields = pd.read_csv("../data/treasury_yields.csv")
yields.columns = ["Date"] + list(yields.columns[1:])
yields["Date"] = pd.to_datetime(yields["Date"])
yields = yields.set_index("Date").reindex(spx_dates).reset_index().rename(columns={"index": "Date"})
yields.ffill(inplace=True)

yields['Slope_10_2'] = yields['GT10 Govt'] - yields['GT2 Govt']
yields['Slope_30_10'] = yields['GT30 Govt'] - yields['GT10 Govt']

yields['YieldChange'] = yields['GT10 Govt']- yields['GT10 Govt'].shift(1) 
yields['YieldCurvature'] = 2*yields['GT10 Govt'] - yields['GT2 Govt'] - yields['GT30 Govt']

#yields.dropna(how='any',inplace=True)

Interest_Rate_Environment_features = yields.copy()

Interest_Rate_Environment_features.to_csv(
    "../data/processed/Interest_Rate_Environment_features.csv",
    index=False
)

print(Interest_Rate_Environment_features.shape)

(6540, 8)


<hr>
Cross-Asset Risk Signals 
<hr>

In [57]:
BondReturn = pd.read_csv("../data/bond_return_proxies.csv")
BondReturn.columns = ["Date"] + list(BondReturn.columns[1:])
BondReturn["Date"] = pd.to_datetime(BondReturn["Date"])
#BondReturn = BondReturn.set_index("Date").reindex(spx_dates).reset_index().rename(columns={"index": "Date"})


Bond_10y = BondReturn[['Date','IEF_proxy']]

Equity_Return = spx[['Date','SPX_return']]
#Equity_Return = Equity_Return.dropna(how='any')


Equity_Bond = Equity_Return.merge(Bond_10y,on='Date',how='inner')

Equity_Bond['Return(Eq-Bond)'] = Equity_Bond['SPX_return'] - Equity_Bond['IEF_proxy'] 
Equity_Bond["Equity_bond_corr"] = Equity_Bond["SPX_return"].rolling(60).corr(Equity_Bond['IEF_proxy'])

CrossAsset_RiskSignals_features = Equity_Bond.copy()

CrossAsset_RiskSignals_features = CrossAsset_RiskSignals_features[[
    "Date",
    "Return(Eq-Bond)",
    "Equity_bond_corr"
]].copy()

CrossAsset_RiskSignals_features.to_csv(
    "../data/processed/CrossAsset_RiskSignals_features.csv",
    index=False
)



print(CrossAsset_RiskSignals_features.shape)

(6540, 3)


In [51]:
CrossAsset_RiskSignals_features

,Date,Return(Eq-Bond),Equity_bond_corr
0,1999-12-31,NaN,NaN
1,2000-01-03,0.001655,NaN
2,2000-01-04,-0.046224,NaN
3,2000-01-05,0.009195,NaN
4,2000-01-06,-0.004295,NaN
...,...,...,...
6535,2025-12-24,0.001041,-0.171719
6536,2025-12-26,-0.000829,-0.183686
6537,2025-12-29,-0.004848,-0.187542
6538,2025-12-30,-0.000477,-0.188428


<hr>
Global Risk Environment Signals 
<hr>

In [34]:
DXY = pd.read_csv("../data/DXY_data.csv")
DXY.columns = ["Date"] + list(DXY.columns[1:])
DXY["Date"] = pd.to_datetime(DXY["Date"])
DXY = DXY.set_index("Date").reindex(spx_dates).reset_index().rename(columns={"index": "Date"})
DXY.ffill(inplace=True)

Equity_Return = spx[['Date', 'SPX_return']].copy()
#Equity_Return = Equity_Return.dropna(how='any')

GlobalRisk = DXY[['Date', 'DXY Curncy']].copy()
GlobalRisk = GlobalRisk.sort_values("Date").reset_index(drop=True)

GlobalRisk["DXY_return"] = np.log(
    GlobalRisk["DXY Curncy"] / GlobalRisk["DXY Curncy"].shift(1)
)

DXY_Equity = Equity_Return.merge(GlobalRisk, on='Date', how='left')
DXY_Equity = DXY_Equity.sort_values("Date").reset_index(drop=True)

DXY_Equity["DXY_mom_63"] = DXY_Equity["DXY Curncy"].pct_change(63)
DXY_Equity["DXY_vol_21"] = DXY_Equity["DXY_return"].rolling(21).std()
DXY_Equity["DXY_SPX_corr"] = (
    DXY_Equity["DXY_return"].rolling(63).corr(DXY_Equity["SPX_return"])
)

Global_RiskEnv_features = DXY_Equity.copy()
Global_RiskEnv_features = Global_RiskEnv_features.drop(columns=["SPX_return"])
Global_RiskEnv_features.to_csv(
    "../data/processed/Global_RiskEnv_features.csv",
    index=False
)

In [35]:
Global_RiskEnv_features

,Date,DXY Curncy,DXY_return,DXY_mom_63,DXY_vol_21,DXY_SPX_corr
0,1999-12-31,101.870,NaN,NaN,NaN,NaN
1,2000-01-03,100.220,-0.016330,NaN,NaN,NaN
2,2000-01-04,100.410,0.001894,NaN,NaN,NaN
3,2000-01-05,100.380,-0.000299,NaN,NaN,NaN
4,2000-01-06,100.480,0.000996,NaN,NaN,NaN
...,...,...,...,...,...,...
6535,2025-12-24,97.976,0.000347,-0.005855,0.002317,0.061570
6536,2025-12-26,98.022,0.000469,-0.001324,0.002172,0.077447
6537,2025-12-29,98.037,0.000153,0.001338,0.002182,0.081047
6538,2025-12-30,98.238,0.002048,0.004735,0.002260,0.081777


<hr>
Economic_Activity_Indicators.
<hr>

In [13]:
import pandas as pd
import numpy as np

# Load ISM
ISM = pd.read_csv("../data/ISM_Manufacturing.csv")
ISM["Date"] = pd.to_datetime(ISM["Date"])

# ISM features
Economic_Activity_Indicators = ISM.copy()
Economic_Activity_Indicators.columns = ["Date", "Industrial_ISM"]
Economic_Activity_Indicators["ISM_change"] = (
    Economic_Activity_Indicators["Industrial_ISM"].diff()
)

Economic_Activity_Indicators = (
    Economic_Activity_Indicators[["Date", "Industrial_ISM", "ISM_change"]]
    .sort_values("Date")
    .reset_index(drop=True)
)

Economic_Activity_Indicators[["Industrial_ISM_lag1", "ISM_change_lag1"]] = (
    Economic_Activity_Indicators[["Industrial_ISM", "ISM_change"]].shift(1)
)

Economic_Activity_Indicators = Economic_Activity_Indicators[["Date", "Industrial_ISM_lag1", "ISM_change_lag1"]]

Economic_Activity_Indicators.to_csv(
    "../data/processed/Economic_Activity_Indicators_features.csv",
    index=False
)

print(Economic_Activity_Indicators.head())

        Date  Industrial_ISM_lag1  ISM_change_lag1
0 1999-12-31                  NaN              NaN
1 2000-01-31                 57.8              NaN
2 2000-02-29                 56.7             -1.1
3 2000-03-31                 55.8             -0.9
4 2000-04-30                 54.9             -0.9


In [12]:
Economic_Activity_Indicators

,Date,Industrial_ISM_lag1,ISM_change_lag1
0,1999-12-31,NaN,NaN
1,2000-01-31,57.8,NaN
2,2000-02-29,56.7,-1.1
3,2000-03-31,55.8,-0.9
4,2000-04-30,54.9,-0.9
...,...,...,...
310,2025-10-31,48.9,0.0
311,2025-11-30,48.8,-0.1
312,2025-12-31,48.0,-0.8
313,2026-01-31,47.9,-0.1


<hr>
Macro_Inflation_Environment
<hr>

In [24]:
import pandas as pd
import numpy as np

# =========================
# Load CPI + Sentiment data
# =========================
CPI_Sentiment = pd.read_csv(
    "../data/CPI_Sentiment.csv",
    header=1
)

CPI_Sentiment = CPI_Sentiment.rename(columns={"Unnamed: 0": "Date"})
CPI_Sentiment["Date"] = pd.to_datetime(CPI_Sentiment["Date"])

# =========================
# Select needed columns
# =========================
Macro_Inflation_Environment = CPI_Sentiment[[
    "Date",
    "CPI Index",
    "CONSSENT Index",
    "CONCCONF Index"
]].copy()

# =========================
# Rename columns
# =========================
Macro_Inflation_Environment = Macro_Inflation_Environment.rename(columns={
    "CPI Index": "CPI_level",
    "CONSSENT Index": "UMich_Sentiment",
    "CONCCONF Index": "CB_Confidence"
})


# =========================
# Convert to numeric
# =========================
for col in ["CPI_level", "UMich_Sentiment", "CB_Confidence"]:
    Macro_Inflation_Environment[col] = pd.to_numeric(
        Macro_Inflation_Environment[col],
        errors="coerce"
    )


Macro_Inflation_Environment = Macro_Inflation_Environment.ffill()

# =========================
# Sort by date
# =========================
Macro_Inflation_Environment = (
    Macro_Inflation_Environment
    .sort_values("Date")
    .reset_index(drop=True)
)

# =========================
# Feature engineering
# =========================
Macro_Inflation_Environment["CPI_change"] = (
    Macro_Inflation_Environment["CPI_level"].diff()
)

Macro_Inflation_Environment["CPI_yoy"] = (
    Macro_Inflation_Environment["CPI_level"].pct_change(12)
)

Macro_Inflation_Environment["CPI_change_3m"] = (
    Macro_Inflation_Environment["CPI_change"].rolling(3).mean()
)

Macro_Inflation_Environment["Inflation_Regime"] = (
    Macro_Inflation_Environment["CPI_yoy"] > 0.03
).astype(int)

# =========================
# Final feature set
# =========================
Macro_Inflation_Environment = Macro_Inflation_Environment[[
    "Date",
    "CPI_change",
    "CPI_yoy",
    "CPI_change_3m",
    "Inflation_Regime",
    "UMich_Sentiment",
    "CB_Confidence"
]].copy()

# =========================
# Apply lag to all features
# =========================
feature_cols = [col for col in Macro_Inflation_Environment.columns if col != "Date"]

Macro_Inflation_Environment[feature_cols] = (
    Macro_Inflation_Environment[feature_cols].shift(1)
)

# =========================
# Rename with _lag1
# =========================
Macro_Inflation_Environment = Macro_Inflation_Environment.rename(columns={
    "CPI_change": "CPI_change_lag1",
    "CPI_yoy": "CPI_yoy_lag1",
    "CPI_change_3m": "CPI_change_3m_lag1",
    "Inflation_Regime": "Inflation_Regime_lag1",
    "UMich_Sentiment": "UMich_Sentiment_lag1",
    "CB_Confidence": "CB_Confidence_lag1"
})

# =========================
# Save
# =========================
Macro_Inflation_Environment.to_csv(
    "../data/processed/Macro_Inflation_Environment_features.csv",
    index=False
)

print(Macro_Inflation_Environment.head())
print(Macro_Inflation_Environment.shape)

        Date  CPI_change_lag1  CPI_yoy_lag1  CPI_change_3m_lag1  \
0 1999-12-31              NaN           NaN                 NaN   
1 2000-01-31              NaN           NaN                 NaN   
2 2000-02-29              0.5           NaN                 NaN   
3 2000-03-31              0.7           NaN                 NaN   
4 2000-04-30              1.0           NaN            0.733333   

   Inflation_Regime_lag1  UMich_Sentiment_lag1  CB_Confidence_lag1  
0                    NaN                   NaN                 NaN  
1                    0.0                 105.4               141.7  
2                    0.0                 112.0               144.7  
3                    0.0                 111.3               140.8  
4                    0.0                 107.1               137.1  
(315, 7)


<hr>
Liquidity / Monetary Conditions 
<hr>

In [19]:
import pandas as pd
import numpy as np

# =========================
# Load M2 data
# =========================

M2 = pd.read_csv("../data/M2.csv")
M2["Date"] = pd.to_datetime(M2["Date"])

# =========================
# Select column (robust)
# =========================

# Automatically detect M2 column (assuming only one besides Date)
m2_col = [col for col in M2.columns if col != "Date"][0]

M2_features = M2[["Date", m2_col]].copy()
M2_features = M2_features.rename(columns={m2_col: "M2_level"})

# =========================
# Sort
# =========================

M2_features = M2_features.sort_values("Date").reset_index(drop=True)

# =========================
# Feature engineering
# =========================

# 1. Basic growth
M2_features["M2_growth"] = M2_features["M2_level"].pct_change()

# 2. 3-month growth
M2_features["M2_growth_3m"] = M2_features["M2_level"].pct_change(3)

# 3. YoY growth
M2_features["M2_growth_yoy"] = M2_features["M2_level"].pct_change(12)


# =========================
# Final feature set
# =========================

M2_features = M2_features[[
    "Date",
    "M2_growth",
    "M2_growth_3m",
    "M2_growth_yoy"]].copy()


# =========================
# Apply lag to all features
# =========================

feature_cols = [col for col in M2_features.columns if col != "Date"]

M2_features[feature_cols] = (
    M2_features[feature_cols].shift(1)
)

# =========================
# Rename with _lag1
# =========================

M2_features = M2_features.rename(columns={
    "M2_growth": "M2_growth_lag1",
    "M2_growth_3m": "M2_growth_3m_lag1",
    "M2_growth_yoy": "M2_growth_yoy_lag1"})

# =========================
# Save
# =========================

M2_features.to_csv(
    "../data/processed/Monetary_Conditions_features.csv",
    index=False
)

print(M2_features.head())
print(M2_features.shape)

        Date  M2_growth_lag1  M2_growth_3m_lag1  M2_growth_yoy_lag1
0 1999-12-31             NaN                NaN                 NaN
1 2000-01-31             NaN                NaN                 NaN
2 2000-02-29        0.006100                NaN                 NaN
3 2000-03-31        0.002849                NaN                 NaN
4 2000-04-30        0.006580           0.015606                 NaN
(314, 4)


<hr>
Safe Haven 
<hr>
    

In [38]:
idx = pd.read_csv("../data/index_prices.csv", header=1)
idx.columns = ["Date"] + list(idx.columns[1:])

print("Columns in index_prices.csv:")
print(idx.columns.tolist())

spx = idx[["Date", "SPX Index"]].copy()
gld = idx[["Date", "GOLDLNPM Index"]].copy()

Safe_Haven = spx.merge(gld,on='Date',how='inner')
Safe_Haven["Date"] = pd.to_datetime(Safe_Haven["Date"])

Safe_Haven = Safe_Haven.set_index("Date").reindex(spx_dates).reset_index().rename(columns={"index": "Date"})
Safe_Haven = Safe_Haven.ffill()

Safe_Haven['GLD_Return'] = np.log(
    Safe_Haven["GOLDLNPM Index"] / Safe_Haven["GOLDLNPM Index"].shift(1)
)
Safe_Haven['GLD_SPX_Ratio'] = Safe_Haven["GOLDLNPM Index"]/ Safe_Haven["SPX Index"]
Safe_Haven['GLD_SPX_Ratio_Return'] = np.log(
    Safe_Haven["GLD_SPX_Ratio"] / Safe_Haven["GLD_SPX_Ratio"].shift(1)
)

Safe_Haven["GLD_SPX_Mom_21"] = Safe_Haven["GLD_SPX_Ratio"].pct_change(21)
Safe_Haven["GLD_SPX_Vol_21"] = Safe_Haven["GLD_SPX_Ratio_Return"].rolling(21).std()



Safe_Haven_features = Safe_Haven[[
    "Date",
    "GLD_Return",
    "GLD_SPX_Ratio",
    "GLD_SPX_Ratio_Return",
    "GLD_SPX_Mom_21",
    "GLD_SPX_Vol_21"
]].copy()


# =========================
# Save
# =========================

Safe_Haven_features.to_csv(
    "../data/processed/Safe_Haven_features.csv",
    index=False
)

print(Safe_Haven_features.head())
print(Safe_Haven_features.shape)



Columns in index_prices.csv:
['Date', 'SPX Index', 'RLV Index', 'RU20INTR Index', 'RTY Index', 'LUACTRUU Index', 'LF98TRUU Index', 'GOLDLNPM Index']
        Date  GLD_Return  GLD_SPX_Ratio  GLD_SPX_Ratio_Return  GLD_SPX_Mom_21  \
0 1999-12-31         NaN            NaN                   NaN             NaN   
1 2000-01-03         NaN            NaN                   NaN             NaN   
2 2000-01-04         NaN       0.201155                   NaN             NaN   
3 2000-01-05   -0.003737       0.200020             -0.005657             NaN   
4 2000-01-06   -0.003751       0.199081             -0.004706             NaN   

   GLD_SPX_Vol_21  
0             NaN  
1             NaN  
2             NaN  
3             NaN  
4             NaN  
(6540, 6)


<hr>
Commodity_Inflation_Shock_Features
<hr>

In [40]:
import pandas as pd
import numpy as np


oil = pd.read_csv("../data/Brent_oil.csv")
oil["Date"] = pd.to_datetime(oil["Date"])
oil = oil.set_index("Date").reindex(spx_dates).reset_index().rename(columns={"index": "Date"})
oil.ffill(inplace=True)
oil = oil.rename(columns={"PX_LAST": "Brent_oil"})

oil["Oil_Return"] = np.log(oil["Brent_oil"]/ oil["Brent_oil"].shift(1))
oil["Oil_Vol_21"] = oil["Oil_Return"].rolling(21).std()
oil["Oil_Mom_63"] = oil["Brent_oil"].pct_change(63)

Commodity_Inflation_Shock_Features = oil[[
    "Date",
    "Oil_Return",
    "Oil_Vol_21",
    "Oil_Mom_63"
]].copy()


Commodity_Inflation_Shock_Features.to_csv(
    "../data/processed/Commodity_Inflation_Shock_Features.csv",
    index=False
)

print(Commodity_Inflation_Shock_Features.head())
print(Commodity_Inflation_Shock_Features.shape)


        Date  Oil_Return  Oil_Vol_21  Oil_Mom_63
0 1999-12-31         NaN         NaN         NaN
1 2000-01-03         NaN         NaN         NaN
2 2000-01-04         NaN         NaN         NaN
3 2000-01-05   -0.027433         NaN         NaN
4 2000-01-06   -0.004646         NaN         NaN
(6540, 4)


<hr>
bond_features
<hr>

In [42]:
BondReturn = pd.read_csv("../data/bond_return_proxies.csv")
BondReturn.columns = ["Date"] + list(BondReturn.columns[1:])
BondReturn["Date"] = pd.to_datetime(BondReturn["Date"])
BondReturn = BondReturn.set_index("Date").reindex(spx_dates).reset_index().rename(columns={"index": "Date"})
BondReturn.ffill(inplace=True)

Bond_10y = BondReturn[['Date','IEF_proxy']]
Bond_10y = Bond_10y.rename(columns={"IEF_proxy": "Bond_10y_return"})
Bond_10y["Bond_10y_Vol_21"] = Bond_10y["Bond_10y_return"].rolling(21).std()

bond_features = Bond_10y.copy()


bond_features.to_csv(
    "../data/processed/bond_Features.csv",
    index=False
)

print(bond_features.head())
print(bond_features.shape)



        Date  Bond_10y_return  Bond_10y_Vol_21
0 1999-12-31              NaN              NaN
1 2000-01-03        -0.011250              NaN
2 2000-01-04         0.007125              NaN
3 2000-01-05        -0.007275              NaN
4 2000-01-06         0.005250              NaN
(6540, 3)


<hr>
Credit Market Stress 
<hr>

In [47]:
CreditSpread = pd.read_csv("../data/credit_spreads.csv",header=1)

CreditSpread = CreditSpread.rename(columns={"Unnamed: 0": "Date","LUAS Index":"IG_Spread","LF98OAS Index":"HY_Spread"})
CreditSpread["Date"] = pd.to_datetime(CreditSpread["Date"])
CreditSpread = CreditSpread.set_index("Date").reindex(spx_dates).reset_index().rename(columns={"index": "Date"})
CreditSpread = CreditSpread.ffill()

CreditSpread["HY_Change"] = CreditSpread["HY_Spread"].diff()
CreditSpread["IG_Change"] = CreditSpread["IG_Spread"].diff()
CreditSpread["HY_IG_Spread"] = CreditSpread["HY_Spread"] - CreditSpread["IG_Spread"]

credit_features = CreditSpread.copy()


credit_features.to_csv(
    "../data/processed/credit_features.csv",
    index=False
)

print(credit_features.head())
print(credit_features.shape)



        Date  IG_Spread  HY_Spread  HY_Change  IG_Change  HY_IG_Spread
0 1999-12-31       1.11       4.67        NaN        NaN          3.56
1 2000-01-03       1.11       4.67        0.0        0.0          3.56
2 2000-01-04       1.11       4.67        0.0        0.0          3.56
3 2000-01-05       1.11       4.67        0.0        0.0          3.56
4 2000-01-06       1.11       4.67        0.0        0.0          3.56
(6540, 6)


<hr>
Recession Indicator Features
<hr>

In [44]:
Recession_Indicator = pd.read_csv("../data/NBER_US.csv")
Recession_Indicator = Recession_Indicator.rename(columns={"PX_LAST": "NBER_US_lag1"})
Recession_Indicator['NBER_US_lag1'] = Recession_Indicator['NBER_US_lag1'].shift(1)

Recession_Indicator_features = Recession_Indicator.copy()

Recession_Indicator_features.to_csv(
    "../data/processed/Recession_Indicator_Features.csv",
    index=False
)

print(Recession_Indicator_features.head())
print(Recession_Indicator_features.shape)



         Date  NBER_US_lag1
0  1999-12-31           NaN
1  2000-01-31           0.0
2  2000-02-29           0.0
3  2000-03-31           0.0
4  2000-04-30           0.0
(315, 2)


In [1]:
## final bucket

In [53]:
import pandas as pd

data = [

    # =========================
    # Equity Market Dynamics
    # =========================
    ["Equity Market Dynamics","SPX_return","SPX_return","No","Daily","Return","Core",
     "log(SPX_t / SPX_{t-1})","equity_market_dynamics_features.csv"],
    
    ["Equity Market Dynamics","SPX_volatility_20d","SPX_volatility_20d","No","Daily","Volatility","Core",
     "rolling std (20d)","equity_market_dynamics_features.csv"],
    
    ["Equity Market Dynamics","SPX_drawdown","SPX_drawdown","No","Daily","Drawdown","Core",
     "rolling max drawdown","equity_market_dynamics_features.csv"],
    
    ["Equity Market Dynamics","SPX_MA_signal","SPX_MA_signal","No","Daily","Trend","Optional",
     "MA50 - MA200","equity_market_dynamics_features.csv"],
    
    ["Equity Market Dynamics","Momentum Factor","Mom","No","Daily","Factor","Optional",
     "Fama-French momentum","equity_market_dynamics_features.csv"],

    # =========================
    # Volatility
    # =========================
    ["Volatility / Risk","VIX_level","VIX","No","Daily","Level","Core",
     "Raw VIX","Volatility_Risk_Sentiment_features.csv"],
    
    ["Volatility / Risk","VIX_change","VIXChange","No","Daily","Change","Core",
     "log change","Volatility_Risk_Sentiment_features.csv"],
    
    ["Volatility / Risk","VIX_term_structure","VIX3M_minus_VIX","No","Daily","Spread","Optional",
     "VIX3M - VIX","Volatility_Risk_Sentiment_features.csv"],

    # =========================
    # Rates
    # =========================
    ["Rates","Slope_10_2","Slope_10_2","No","Daily","Spread","Core",
     "10Y - 2Y","Interest_Rate_Environment_features.csv"],
    
    ["Rates","Slope_30_10","Slope_30_10","No","Daily","Spread","Optional",
     "30Y - 10Y","Interest_Rate_Environment_features.csv"],
    
    ["Rates","YieldChange","YieldChange","No","Daily","Change","Core",
     "GT10 - lag(GT10)","Interest_Rate_Environment_features.csv"],
    
    ["Rates","YieldCurvature","YieldCurvature","No","Daily","Curvature","Optional",
     "2*10Y - 2Y - 30Y","Interest_Rate_Environment_features.csv"],

    # =========================
    # Cross Asset
    # =========================
    ["Cross Asset","Eq-Bond Spread","Return_Eq_Bond","No","Daily","Spread","Core",
     "SPX - Bond","CrossAsset_RiskSignals_features.csv"],
    
    ["Cross Asset","Eq-Bond Corr","Equity_bond_corr","No","Daily","Correlation","Core",
     "rolling corr","CrossAsset_RiskSignals_features.csv"],

    # =========================
    # FX
    # =========================
    ["FX","DXY_return","DXY_return","No","Daily","Return","Optional",
     "log return","Global_RiskEnv_features.csv"],
    
    ["FX","DXY_momentum","DXY_mom_63","No","Daily","Momentum","Optional",
     "63d","Global_RiskEnv_features.csv"],
    
    ["FX","DXY_vol","DXY_vol_21","No","Daily","Volatility","Optional",
     "21d std","Global_RiskEnv_features.csv"],
    
    ["FX","DXY_SPX_corr","DXY_SPX_corr","No","Daily","Correlation","Optional",
     "corr","Global_RiskEnv_features.csv"],

    # =========================
    # Economic Activity (LAGGED)
    # =========================
    ["Macro Growth","Industrial_ISM","Industrial_ISM_lag1","Yes","Monthly→Daily","Level","Core",
     "Lagged ISM","Economic_Activity_Indicators_features.csv"],
    
    ["Macro Growth","ISM_change","ISM_change_lag1","Yes","Monthly→Daily","Change","Optional",
     "diff + lag","Economic_Activity_Indicators_features.csv"],

    # =========================
    # Inflation (LAGGED)
    # =========================
    ["Inflation","CPI_change","CPI_change_lag1","Yes","Monthly→Daily","Change","Optional",
     "diff + lag","Macro_Inflation_Environment_features.csv"],
    
    ["Inflation","CPI_yoy","CPI_yoy_lag1","Yes","Monthly→Daily","YoY","Core",
     "YoY inflation","Macro_Inflation_Environment_features.csv"],
    
    ["Inflation","CPI_change_3m","CPI_change_3m_lag1","Yes","Monthly→Daily","Smoothed","Optional",
     "3m avg","Macro_Inflation_Environment_features.csv"],
    
    ["Inflation","Inflation_Regime","Inflation_Regime_lag1","Yes","Monthly→Daily","Regime","Core",
     "CPI > 3%","Macro_Inflation_Environment_features.csv"],
    
    ["Inflation","UMich","UMich_Sentiment_lag1","Yes","Monthly→Daily","Level","Core",
     "Lagged sentiment","Macro_Inflation_Environment_features.csv"],
    
    ["Inflation","CB","CB_Confidence_lag1","Yes","Monthly→Daily","Level","Core",
     "Lagged confidence","Macro_Inflation_Environment_features.csv"],

    # =========================
    # Monetary (LAGGED)
    # =========================
    ["Liquidity","M2_growth","M2_growth_lag1","Yes","Monthly→Daily","Change","Optional",
     "pct_change","Monetary_Conditions_features.csv"],
    
    ["Liquidity","M2_growth_3m","M2_growth_3m_lag1","Yes","Monthly→Daily","Smoothed","Optional",
     "3m","Monetary_Conditions_features.csv"],
    
    ["Liquidity","M2_growth_yoy","M2_growth_yoy_lag1","Yes","Monthly→Daily","YoY","Core",
     "YoY","Monetary_Conditions_features.csv"],

    # =========================
    # Safe Haven
    # =========================
    ["Safe Haven","Gold_return","GLD_Return","No","Daily","Return","Optional",
     "log return","Safe_Haven_features.csv"],
    
    ["Safe Haven","Gold/SPX","GLD_SPX_Ratio","No","Daily","Ratio","Core",
     "Gold / SPX","Safe_Haven_features.csv"],
    
    ["Safe Haven","Gold/SPX Vol","GLD_SPX_Vol_21","No","Daily","Volatility","Optional",
     "21d std","Safe_Haven_features.csv"],

    # =========================
    # Commodity
    # =========================
    ["Commodity","Oil_return","Oil_Return","No","Daily","Return","Optional",
     "log return","Commodity_Inflation_Shock_Features.csv"],
    
    ["Commodity","Oil_vol","Oil_Vol_21","No","Daily","Volatility","Optional",
     "21d","Commodity_Inflation_Shock_Features.csv"],
    
    ["Commodity","Oil_mom","Oil_Mom_63","No","Daily","Momentum","Optional",
     "63d","Commodity_Inflation_Shock_Features.csv"],

    # =========================
    # Credit
    # =========================
    ["Credit","IG Spread","IG_Spread","No","Daily","Level","Core",
     "IG spread","credit_features.csv"],
    
    ["Credit","HY Spread","HY_Spread","No","Daily","Level","Core",
     "HY spread","credit_features.csv"],
    
    ["Credit","HY Change","HY_Change","No","Daily","Change","Core",
     "diff","credit_features.csv"],
    
    ["Credit","IG Change","IG_Change","No","Daily","Change","Optional",
     "diff","credit_features.csv"],
    

    # =========================
    # Label
    # =========================
    ["Label","NBER_Recession","NBER_Recession","No","Monthly","Label","Evaluation",
     "Do NOT use in ML","Recession_Indicator_Features.csv"],
]

columns = [
    "Bucket",
    "Feature_Name",
    "Final_Name",
    "Lagged",
    "Frequency",
    "Feature_Type",
    "Use",
    "Description",
    "Processed_File"
]

catalog = pd.DataFrame(data, columns=columns)

#catalog.to_excel("feature_catalog_FINAL.xlsx", index=False)

print(catalog.head())

                   Bucket        Feature_Name          Final_Name Lagged  \
0  Equity Market Dynamics          SPX_return          SPX_return     No   
1  Equity Market Dynamics  SPX_volatility_20d  SPX_volatility_20d     No   
2  Equity Market Dynamics        SPX_drawdown        SPX_drawdown     No   
3  Equity Market Dynamics       SPX_MA_signal       SPX_MA_signal     No   
4  Equity Market Dynamics     Momentum Factor                 Mom     No   

  Frequency Feature_Type       Use             Description  \
0     Daily       Return      Core  log(SPX_t / SPX_{t-1})   
1     Daily   Volatility      Core       rolling std (20d)   
2     Daily     Drawdown      Core    rolling max drawdown   
3     Daily        Trend  Optional            MA50 - MA200   
4     Daily       Factor  Optional    Fama-French momentum   

                        Processed_File  
0  equity_market_dynamics_features.csv  
1  equity_market_dynamics_features.csv  
2  equity_market_dynamics_features.csv  
3  equ

In [54]:
catalog

,Bucket,Feature_Name,Final_Name,Lagged,Frequency,Feature_Type,Use,Description,Processed_File
0,Equity Market Dynamics,SPX_return,SPX_return,No,Daily,Return,Core,log(SPX_t / SPX_{t-1}),equity_market_dynamics_features.csv
1,Equity Market Dynamics,SPX_volatility_20d,SPX_volatility_20d,No,Daily,Volatility,Core,rolling std (20d),equity_market_dynamics_features.csv
2,Equity Market Dynamics,SPX_drawdown,SPX_drawdown,No,Daily,Drawdown,Core,rolling max drawdown,equity_market_dynamics_features.csv
3,Equity Market Dynamics,SPX_MA_signal,SPX_MA_signal,No,Daily,Trend,Optional,MA50 - MA200,equity_market_dynamics_features.csv
4,Equity Market Dynamics,Momentum Factor,Mom,No,Daily,Factor,Optional,Fama-French momentum,equity_market_dynamics_features.csv
5,Volatility / Risk,VIX_level,VIX,No,Daily,Level,Core,Raw VIX,Volatility_Risk_Sentiment_features.csv
6,Volatility / Risk,VIX_change,VIXChange,No,Daily,Change,Core,log change,Volatility_Risk_Sentiment_features.csv
7,Volatility / Risk,VIX_term_structure,VIX3M_minus_VIX,No,Daily,Spread,Optional,VIX3M - VIX,Volatility_Risk_Sentiment_features.csv
8,Rates,Slope_10_2,Slope_10_2,No,Daily,Spread,Core,10Y - 2Y,Interest_Rate_Environment_features.csv
9,Rates,Slope_30_10,Slope_30_10,No,Daily,Spread,Optional,30Y - 10Y,Interest_Rate_Environment_features.csv
